In [19]:
import importlib
import setup
from steering_extraction import extractAllLayer, generateSteering
import torch

import steering_extraction

importlib.reload(setup)
importlib.reload(steering_extraction)

<module 'steering_extraction' from '/workspace/Dissertation_Project/steering_extraction.py'>

In [13]:
anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=50)

In [15]:
neutral_statement[:5]

['Just invite him in on opening day and close the door, seal it up for 2000 years and that will be the end of the matter.',
 'Of the 7’1 jacked guy. Yes',
 'Its acts as a moisture barrier so the bread wont get soggy no matter what you put in it',
 'hard to bound with half team who leave after every season',
 "Which state? It's actually quite rare."]

In [1]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                Size  Used Avail Use% Mounted on
mfs#euro.runpod.net:9421  2.3P  1.7P  662T  72% /workspace


In [3]:
model,tokenizer = setup.modelSetup()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [20]:
prompt = "How to make my girlfriend happy in long distance relationship?"
text_baseline = generateSteering(
    user_text=prompt,
    system_text="Ensure your response does not exceed 50 words",
    # steering_vector=
    model=model,
    tokenizer=tokenizer
)
print(text_baseline)

To make your girlfriend happy in a long-distance relationship, communicate regularly, set a shared calendar, and schedule video calls. Send surprise gifts, care packages, or love notes. Plan visits or trips to see each other. Show interest in her daily life, and be open to feedback and affection.


In [ ]:
hidden_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vectors = {
    'anger': torch.zeros(hidden_layers,hidden_size),
    'happiness': torch.zeros(hidden_layers,hidden_size),
    'sadness': torch.zeros(hidden_layers,hidden_size),
    'love': torch.zeros(hidden_layers,hidden_size),
    'fear': torch.zeros(hidden_layers,hidden_size)
}

counts = {
    'neutral': 0,
    'anger': 0,
    'happiness': 0,
    'sadness': 0,
    'love': 0,
    'fear': 0
}
# Emotion text dataset
emotion_sets = {
    # 'neutral': expanded_neutral_texts,
    'anger': anger_statement,
    'happiness': happiness_statement,
    'sadness': sadness_statement,
    'love': love_statement,
    'fear': fear_statement
}
# For plotting purposes
emotion_vectors = {
    # 'neutral': [],
    'anger': [],
    'happiness': [],
    'sadness': [],
    'love': [],
    'fear': []
}


In [ ]:
layer_id  = 21
for emotion, prompts in emotion_sets.items():
    for prompt in prompts:
        vector = extractAllLayer(prompt, model, tokenizer) # [32,4096]
        # WHY DID I TAKE THE MEAN OF A VECTOR OF SHAPED [1,4096]
        emotion_vectors[emotion].append(vector[layer_id].cpu().numpy())
        vectors[emotion] += vector
        counts[emotion] += 1

# Normalize vectors to get means
for emotion in vectors:
    vectors[emotion] = vectors[emotion] / counts[emotion]